In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import sqlite3
import numpy as np

# Constants
BASE_URL = 'http://books.toscrape.com/'
CURRENCY_CONVERSION_RATE = 105.50 # 1 GBP = 105.50 INR, as per project requirements

# Function to fetch HTML content of a URL
def get_page_html(url):
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status() # Raise HTTPError for bad responses (4xx or 5xx)
        return response.text
    except requests.exceptions.RequestException as e:
        print(f"Error fetching {url}: {e}")
        return None

# Function to parse a book's details from a product article tag
def parse_book_data(book_article):
    title = book_article.h3.a['title']
    price_str = book_article.find('p', class_='price_color').text
    star_rating_text = book_article.find('p', class_='star-rating')['class'][1]
    availability_text = book_article.find('p', class_='instock availability').text.strip()

    return {
        'title': title,
        'price_gbp_raw': price_str,
        'star_rating_text': star_rating_text,
        'availability_text': availability_text
    }

# Function to get category links from the main page
def get_category_links(base_url):
    html = get_page_html(base_url)
    if not html:
        return []
    soup = BeautifulSoup(html, 'html.parser')
    category_list = soup.find('ul', class_='nav').find('ul')
    category_links = []
    if category_list:
        for li in category_list.find_all('li'):
            link = li.a['href']
            name = li.a.text.strip()
            # Exclude 'Books' (All products) category as we want specific ones
            if name != 'Books':
                category_links.append({'name': name, 'url': base_url + link})
    return category_links

In [3]:
# Main scraping function to collect data across categories and pages
def scrape_books_data(base_url, num_categories=3):
    all_books_data = []
    category_links = get_category_links(base_url)

    # Scrape at least `num_categories` or all if less than `num_categories` available
    categories_to_scrape = category_links[:num_categories]

    print(f"Scraping {len(categories_to_scrape)} categories...")

    for category_info in categories_to_scrape:
        category_name = category_info['name']
        category_url = category_info['url'].replace(base_url, '') # Relative path for navigation

        print(f"  Scraping category: {category_name}")

        page_num = 1
        while True:
            # Construct the URL for the current category page
            if 'index.html' in category_url:
                current_page_url = base_url + category_url.replace('index.html', f'page-{page_num}.html')
            else:
                # For categories that might not end in index.html (though books.toscrape does)
                current_page_url = category_info['url'].replace('index.html', '') + f'page-{page_num}.html'

            html = get_page_html(current_page_url)
            if not html:
                print(f"    No more pages or error fetching {current_page_url}")
                break

            soup = BeautifulSoup(html, 'html.parser')
            book_articles = soup.find_all('article', class_='product_pod')

            if not book_articles:
                break # No more books on this page, or no more pages

            for book_article in book_articles:
                book_data = parse_book_data(book_article)
                book_data['category'] = category_name
                all_books_data.append(book_data)

            # Check for next page link
            next_button = soup.find('li', class_='next')
            if not next_button:
                break # No 'next' button, so no more pages

            page_num += 1

    return pd.DataFrame(all_books_data)

# Execute scraping and create raw DataFrame
raw_books_df = scrape_books_data(BASE_URL, num_categories=5) # Scrape 5 categories to ensure >= 60 books

print(f"Scraped {len(raw_books_df)} books.")
print(raw_books_df.head())


Scraping 5 categories...
  Scraping category: Travel
Error fetching http://books.toscrape.com/catalogue/category/books/travel_2/page-1.html: 404 Client Error: Not Found for url: http://books.toscrape.com/catalogue/category/books/travel_2/page-1.html
    No more pages or error fetching http://books.toscrape.com/catalogue/category/books/travel_2/page-1.html
  Scraping category: Mystery
  Scraping category: Historical Fiction
  Scraping category: Sequential Art
  Scraping category: Classics
Error fetching http://books.toscrape.com/catalogue/category/books/classics_6/page-1.html: 404 Client Error: Not Found for url: http://books.toscrape.com/catalogue/category/books/classics_6/page-1.html
    No more pages or error fetching http://books.toscrape.com/catalogue/category/books/classics_6/page-1.html
Scraped 133 books.
                                             title price_gbp_raw  \
0                                    Sharp Objects       Â£47.82   
1                             In a Dark, 

In [4]:
# Data Cleaning and Enrichment

def clean_and_enrich_data(df):
    cleaned_df = df.copy()

    # 1. Clean price_gbp
    def parse_price(price_str):
        try:
            # Remove currency symbol and convert to float
            return float(re.sub(r'[^\d.]', '', price_str))
        except (ValueError, TypeError):
            return np.nan

    cleaned_df['price_gbp'] = cleaned_df['price_gbp_raw'].apply(parse_price)

    # 2. Convert star_rating_text to integer rating
    star_rating_map = {
        'One': 1,
        'Two': 2,
        'Three': 3,
        'Four': 4,
        'Five': 5
    }
    cleaned_df['rating'] = cleaned_df['star_rating_text'].map(star_rating_map)

    # 3. Parse availability_text to boolean in_stock
    def parse_availability(availability_str):
        if 'In stock' in availability_str:
            return True
        elif 'Out of stock' in availability_str:
            return False
        return np.nan # Indicate unparseable

    cleaned_df['in_stock'] = cleaned_df['availability_text'].apply(parse_availability)

    # Handle parsing failures with median imputation for numeric fields and drop for boolean
    # Median imputation for price_gbp and rating
    for col in ['price_gbp', 'rating']:
        if cleaned_df[col].isnull().any():
            median_val = cleaned_df[col].median()
            print(f"Imputing missing values in '{col}' with median: {median_val}")
            cleaned_df[col].fillna(median_val, inplace=True)

    # Drop rows where 'in_stock' could not be parsed (as per README decision)
    initial_rows = len(cleaned_df)
    cleaned_df.dropna(subset=['in_stock'], inplace=True)
    if len(cleaned_df) < initial_rows:
        print(f"Dropped {initial_rows - len(cleaned_df)} rows due to unparseable 'in_stock' values.")

    # Ensure rating is integer type after imputation (if any)
    cleaned_df['rating'] = cleaned_df['rating'].astype(int)
    # Ensure in_stock is integer (boolean stored as 0/1 in SQLite)
    cleaned_df['in_stock'] = cleaned_df['in_stock'].astype(int)

    # 4. Convert price_gbp to price_inr
    cleaned_df['price_inr'] = cleaned_df['price_gbp'] * CURRENCY_CONVERSION_RATE

    # Select and reorder final columns
    final_df = cleaned_df[['title', 'category', 'price_gbp', 'price_inr', 'rating', 'in_stock']]

    return final_df

processed_books_df = clean_and_enrich_data(raw_books_df)

print(f"Processed {len(processed_books_df)} books.")
print(processed_books_df.head())
print(processed_books_df.info())


Processed 133 books.
                                             title category  price_gbp  \
0                                    Sharp Objects  Mystery      47.82   
1                             In a Dark, Dark Wood  Mystery      19.63   
2                              The Past Never Ends  Mystery      56.50   
3                                 A Murder in Time  Mystery      16.64   
4  The Murder of Roger Ackroyd (Hercule Poirot #4)  Mystery      44.10   

   price_inr  rating  in_stock  
0   5045.010       4         1  
1   2070.965       1         1  
2   5960.750       4         1  
3   1755.520       1         1  
4   4652.550       4         1  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 133 entries, 0 to 132
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   title      133 non-null    object 
 1   category   133 non-null    object 
 2   price_gbp  133 non-null    float64
 3   price_inr  133 non-null    

In [5]:
# Database Design and Loading

DB_NAME = 'books.db'

def create_and_load_database(df, db_name):
    conn = sqlite3.connect(db_name)
    cursor = conn.cursor()

    # 1. Create categories table
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS categories (
            category_id INTEGER PRIMARY KEY AUTOINCREMENT,
            category_name TEXT UNIQUE NOT NULL
        )
    ''')

    # Insert unique categories into the categories table
    unique_categories = df['category'].unique()
    for category_name in unique_categories:
        try:
            cursor.execute('INSERT INTO categories (category_name) VALUES (?)', (category_name,))
        except sqlite3.IntegrityError:
            # Category already exists, skip
            pass
    conn.commit()

    # Fetch category_ids to merge with the main dataframe
    categories_df = pd.read_sql_query("SELECT category_id, category_name FROM categories", conn)
    df = pd.merge(df, categories_df, left_on='category', right_on='category_name', how='left')

    # 2. Create books table
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS books (
            book_id INTEGER PRIMARY KEY AUTOINCREMENT,
            title TEXT NOT NULL,
            price_gbp REAL NOT NULL,
            price_inr REAL NOT NULL,
            rating INTEGER NOT NULL,
            in_stock INTEGER NOT NULL,
            category_id INTEGER NOT NULL,
            FOREIGN KEY (category_id) REFERENCES categories(category_id)
        )
    ''')

    # Insert data into the books table using pandas to_sql for convenience
    # Drop the temporary 'category' and 'category_name' columns before inserting into books table
    df_to_load = df[['title', 'price_gbp', 'price_inr', 'rating', 'in_stock', 'category_id']]
    df_to_load.to_sql('books', conn, if_exists='replace', index=False)

    conn.close()
    print(f"Database '{db_name}' created and data loaded successfully.")

create_and_load_database(processed_books_df, DB_NAME)


Database 'books.db' created and data loaded successfully.


In [7]:
# SQL Queries and Pandas Verification

conn = sqlite3.connect(DB_NAME)

# List to store query results for display
query_results = []

print("--- Executing SQL Queries ---")

# Query 1: SELECT books with rating = 5
query1 = "SELECT title, rating FROM books WHERE rating = 5 LIMIT 10;"
print(f"\nQuery 1: {query1}")
df_q1 = pd.read_sql_query(query1, conn)
print(df_q1)
query_results.append({"query": query1, "output": df_q1})

# Query 2: ORDER BY price_inr DESC and LIMIT 5 (top 5 most expensive books)
query2 = "SELECT title, price_inr FROM books ORDER BY price_inr DESC LIMIT 5;"
print(f"\nQuery 2: {query2}")
df_q2 = pd.read_sql_query(query2, conn)
print(df_q2)
query_results.append({"query": query2, "output": df_q2})

# Query 3: DISTINCT categories
query3 = "SELECT DISTINCT category_name FROM categories;"
print(f"\nQuery 3: {query3}")
df_q3 = pd.read_sql_query(query3, conn)
print(df_q3)
query_results.append({"query": query3, "output": df_q3})

# Query 4: Books with price_gbp BETWEEN 20 and 30
query4 = "SELECT title, price_gbp FROM books WHERE price_gbp BETWEEN 20.0 AND 30.0 LIMIT 10;"
print(f"\nQuery 4: {query4}")
df_q4 = pd.read_sql_query(query4, conn)
print(df_q4)
query_results.append({"query": query4, "output": df_q4})

# Query 5: JOIN - List book titles with their category names (top 10 by rating and price)
query5 = """
SELECT b.title, c.category_name, b.rating, b.price_gbp
FROM books b
JOIN categories c ON b.category_id = c.category_id
ORDER BY b.rating DESC, b.price_gbp DESC
LIMIT 10;
"""
print(f"\nQuery 5 (JOIN): {query5}")
df_q5_sql = pd.read_sql_query(query5, conn)
print("SQL Query Result:")
print(df_q5_sql)
query_results.append({"query": query5, "output": df_q5_sql})

print("\n--- Pandas Verification for JOIN query ---")

# Read both tables into pandas DataFrames
df_books = pd.read_sql_query("SELECT * FROM books", conn)
df_categories = pd.read_sql_query("SELECT * FROM categories", conn)

# Reproduce the JOIN query using pd.merge
df_merged = pd.merge(df_books, df_categories, on='category_id', how='inner')

# Apply the same filtering and sorting as in Query 5
df_merged_pandas = df_merged.sort_values(by=['rating', 'price_gbp'], ascending=[False, False]) \
                              .head(10)[['title', 'category_name', 'rating', 'price_gbp']]

# Reset index for proper comparison with SQL query result
df_merged_pandas = df_merged_pandas.reset_index(drop=True)

print("Pandas merge result:")
print(df_merged_pandas)

# Verify if the results are equivalent
comparison_result = df_q5_sql.equals(df_merged_pandas)
print(f"\nAre SQL JOIN and Pandas merge results equivalent? {comparison_result}")

conn.close()


--- Executing SQL Queries ---

Query 1: SELECT title, rating FROM books WHERE rating = 5 LIMIT 10;
                                               title  rating
0             A Time of Torment (Charlie Parker #14)       5
1  What Happened on Beale Street (Secrets of the ...       5
2  The Bachelor Girl's Guide to Murder (Herringfo...       5
3                  The Silkworm (Cormoran Strike #2)       5
4                                  The Girl You Lost       5
5            A Flight of Arrows (The Pathfinders #2)       5
6                                       Mrs. Houdini       5
7                              The Passion of Dolssa       5
8                             Voyager (Outlander #3)       5
9                                       The Red Tent       5

Query 2: SELECT title, price_inr FROM books ORDER BY price_inr DESC LIMIT 5;
                                               title  price_inr
0                      Boar Island (Anna Pigeon #19)    6275.14
1  The No. 1 Ladies' Det